# Weather Hazard Model Rough Work

This notebook is a quick benchmark to check whether non-linear classifiers improve the +7 day Weather Hazard Category task. It keeps the same WHC target creation and one-day lag convention as the classification experiment, then compares Random Forest, XGBoost, LightGBM and MLP across multiple SMOTE strategies.


In [9]:
import warnings

import adv_ml_at2 as at2
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from adv_ml_at2.modeling.custom_models import make_smote_classifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1. Load Data and Create WHC Target


In [10]:
additional_hourly_variables = [
    "pressure_msl",
    "dew_point_2m",
    "apparent_temperature",
    "shortwave_radiation",
    "sunshine_duration",
]
daily_variables = ["weather_code", "wind_direction_10m_dominant", "daylight_duration"]

hourly_path = at2.dataset.RAW_DATA_DIR / "sydney_weather_hourly.csv"
daily_path = at2.dataset.RAW_DATA_DIR / "sydney_weather_daily.csv"

if hourly_path.exists() and daily_path.exists():
    hourly_weather_df = pd.read_csv(hourly_path, parse_dates=["time"])
    daily_weather_df = pd.read_csv(daily_path, parse_dates=["time"])
else:
    hourly_weather_df, daily_weather_df = at2.dataset.collect_historical_weather(
        start_date="2000-01-01",
        end_date="2025-12-31",
        additional_hourly_variables=additional_hourly_variables,
        daily_variables=daily_variables,
    )
    at2.dataset.save_weather_dataframes(hourly_weather_df, daily_weather_df)

aggregation_rules = {
    "temperature_2m": ["mean", "max", "min", "median"],
    "relative_humidity_2m": ["mean", "max", "min", "median"],
    "wind_speed_10m": ["mean", "max"],
    "wind_gusts_10m": ["mean", "max"],
    "cloud_cover": ["mean", "max"],
    "precipitation": ["sum", "max"],
    "snowfall": ["sum", "max"],
    "pressure_msl": ["mean", "max", "min"],
    "dew_point_2m": ["mean", "max", "min"],
    "apparent_temperature": ["mean", "max", "min"],
    "shortwave_radiation": ["mean", "max"],
    "sunshine_duration": ["sum"],
}

daily_features_df = at2.dataset.build_daily_weather_dataset(
    hourly_weather_df=hourly_weather_df,
    daily_weather_df=daily_weather_df,
    aggregation_rules=aggregation_rules,
)

forecast_horizon_days = 7
target_name = "whc_target_7d"
target_label_name = "whc_label_target_7d"

daily_whi_df = at2.features.aggregate_daily_whi(hourly_weather_df)
hazard_columns = [
    "rain_hazard",
    "wind_hazard",
    "cloud_hazard",
    "snow_hazard",
    "temperature_hazard",
    "whi",
    "whc",
    "whc_label",
]

weather_df = daily_features_df.merge(
    daily_whi_df[["time"] + hazard_columns],
    on="time",
    how="inner",
    validate="one_to_one",
)
weather_df = at2.features.create_whc_forecast_targets(
    weather_df,
    horizon=forecast_horizon_days,
)
weather_df = at2.features.lag_feature_space(
    weather_df,
    target_columns=[target_name, target_label_name],
    lag_days=1,
)
weather_df = weather_df.dropna(subset=[target_name]).reset_index(drop=True)
weather_df[target_name] = weather_df[target_name].astype("int64")

print(weather_df.shape)
display(weather_df[["time", "whi_lag_1d", "whc_lag_1d", target_name]].tail())
display(weather_df[target_name].value_counts().sort_index().rename("count").to_frame())


(9489, 44)


,time,whi_lag_1d,whc_lag_1d,whc_target_7d
9484,2025-12-20,21.653333,0.0,0
9485,2025-12-21,34.698333,1.0,1
9486,2025-12-22,29.688333,1.0,0
9487,2025-12-23,34.285000,1.0,0
9488,2025-12-24,28.305000,1.0,1


,count
whc_target_7d,
0,5239
1,3965
2,274
3,11


## 2. Prepare Modelling Matrix


In [11]:
features_list = [
    "precipitation_sum_lag_1d",
    "wind_gusts_10m_max_lag_1d",
    "cloud_cover_mean_lag_1d",
    "temperature_2m_mean_lag_1d",
    "rain_hazard_lag_1d",
    "wind_hazard_lag_1d",
    "cloud_hazard_lag_1d",
    "temperature_hazard_lag_1d",
    "whi_lag_1d",
    "whc_lag_1d",
]

model_df = weather_df[["time"] + features_list + [target_name]].copy()
model_df["precipitation_sum_log1p_lag_1d"] = np.log1p(model_df["precipitation_sum_lag_1d"])
model_df = model_df.drop(columns=["precipitation_sum_lag_1d"])

day_of_year = model_df["time"].dt.dayofyear
days_in_year = np.where(model_df["time"].dt.is_leap_year, 366, 365)
angle = 2 * np.pi * (day_of_year - 1) / days_in_year
model_df["day_of_year_sin"] = np.sin(angle)
model_df["day_of_year_cos"] = np.cos(angle)

for lag in [3, 7, 14, 30]:
    model_df[f"whi_lag_{lag}d"] = model_df["whi_lag_1d"].shift(lag - 1)

for window in [7, 30]:
    rolling = model_df["whi_lag_1d"].rolling(window=window, min_periods=window)
    model_df[f"whi_rolling_mean_{window}d"] = rolling.mean()
    model_df[f"whi_rolling_std_{window}d"] = rolling.std(ddof=0)

model_df = model_df.dropna().reset_index(drop=True)

train_end_date = pd.Timestamp("2021-12-31")
validation_end_date = pd.Timestamp("2023-12-31")
train_mask = model_df["time"].le(train_end_date)
val_mask = model_df["time"].gt(train_end_date) & model_df["time"].le(validation_end_date)
test_mask = model_df["time"].gt(validation_end_date)

feature_columns = [column for column in model_df.columns if column not in ["time", target_name]]
X_train = model_df.loc[train_mask, feature_columns].copy()
X_val = model_df.loc[val_mask, feature_columns].copy()
X_test = model_df.loc[test_mask, feature_columns].copy()
y_train = model_df.loc[train_mask, target_name].copy()
y_val = model_df.loc[val_mask, target_name].copy()
y_test = model_df.loc[test_mask, target_name].copy()

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "start": [model_df.loc[train_mask, "time"].min(), model_df.loc[val_mask, "time"].min(), model_df.loc[test_mask, "time"].min()],
    "end": [model_df.loc[train_mask, "time"].max(), model_df.loc[val_mask, "time"].max(), model_df.loc[test_mask, "time"].max()],
}, index=["train", "validation", "test"])
display(split_summary)

display(pd.concat({
    "train": y_train.value_counts().sort_index(),
    "validation": y_val.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}, axis=1).fillna(0).astype(int))
print("Feature count:", len(feature_columns))


,rows,start,end
train,8006,2000-01-31,2021-12-31
validation,730,2022-01-01,2023-12-31
test,724,2024-01-01,2025-12-24


,train,validation,test
whc_target_7d,,,
0,4561,353,312
1,3234,341,375
2,203,34,36
3,8,2,1


Feature count: 20


## 3. Random Search Followed by Local Optuna Search

Each model family first gets a broad random search. The best random-search configuration for that model is then used as the centre of a narrower Optuna search. This keeps the rough notebook fast and practical: random search explores the wider space, while Optuna spends its trials around a promising region. The MLP uses a `StandardScaler` inside its estimator pipeline because neural networks are scale-sensitive, while the tree models use the raw prepared matrix.


In [12]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

class_ids = sorted(at2.features.WHC_LABELS)
class_names = [at2.features.WHC_LABELS[class_id] for class_id in class_ids]
smote_strategies = ["none", "mild", "moderate", "strong", "auto"]
mlp_architectures = {
    "32": (32,),
    "64": (64,),
    "64_32": (64, 32),
    "128_64": (128, 64),
}


def score_classifier(model, X, y):
    predictions = model.predict(X)
    return {
        "macro_recall": recall_score(y, predictions, average="macro", zero_division=0),
        "macro_f1": f1_score(y, predictions, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "accuracy": accuracy_score(y, predictions),
    }


def random_choice(options, rng):
    return options[int(rng.integers(0, len(options)))]


def random_log_float(low, high, rng):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))


def random_log_int(low, high, rng):
    return int(np.clip(round(random_log_float(low, high, rng)), low, high))


def categorical_window(value, options, radius=1):
    if value not in options:
        return options
    index = options.index(value)
    lower = max(0, index - radius)
    upper = min(len(options), index + radius + 1)
    return options[lower:upper]


def int_bounds_around(value, lower, upper, radius):
    value = int(value)
    return max(lower, value - radius), min(upper, value + radius)


def float_bounds_around(value, lower, upper, radius):
    value = float(value)
    return max(lower, value - radius), min(upper, value + radius)


def log_bounds_around(value, lower, upper, factor=3.0):
    value = float(value)
    return max(lower, value / factor), min(upper, value * factor)


def sample_random_model_params(model_name, rng):
    params = {
        "sampling_strategy": random_choice(smote_strategies, rng),
        "k_neighbors": int(rng.integers(1, 8)),
    }

    if model_name == "RandomForest":
        params.update({
            "n_estimators": int(random_choice(list(range(200, 901, 100)), rng)),
            "max_depth": random_choice([3, 5, 8, 12, None], rng),
            "min_samples_leaf": random_log_int(1, 50, rng),
            "min_samples_split": random_log_int(2, 100, rng),
            "max_features": random_choice(["sqrt", "log2", 0.5, None], rng),
            "bootstrap": bool(random_choice([True, False], rng)),
        })
    elif model_name == "XGBoost":
        params.update({
            "n_estimators": int(random_choice(list(range(100, 901, 100)), rng)),
            "max_depth": int(rng.integers(2, 9)),
            "learning_rate": random_log_float(0.01, 0.20, rng),
            "min_child_weight": random_log_float(1.0, 20.0, rng),
            "subsample": float(rng.uniform(0.60, 1.00)),
            "colsample_bytree": float(rng.uniform(0.60, 1.00)),
            "gamma": float(rng.uniform(0.0, 5.0)),
            "reg_alpha": random_log_float(1e-4, 10.0, rng),
            "reg_lambda": random_log_float(1e-2, 50.0, rng),
        })
    elif model_name == "LightGBM":
        params.update({
            "n_estimators": int(random_choice(list(range(100, 901, 100)), rng)),
            "learning_rate": random_log_float(0.01, 0.20, rng),
            "num_leaves": int(rng.integers(7, 64)),
            "max_depth": random_choice([2, 3, 4, 5, 7, -1], rng),
            "min_child_samples": int(rng.integers(20, 151)),
            "subsample": float(rng.uniform(0.60, 1.00)),
            "colsample_bytree": float(rng.uniform(0.60, 1.00)),
            "reg_alpha": random_log_float(1e-4, 10.0, rng),
            "reg_lambda": random_log_float(1e-2, 50.0, rng),
        })
    elif model_name == "MLP":
        params.update({
            "hidden_layer_sizes": random_choice(list(mlp_architectures), rng),
            "activation": random_choice(["relu", "tanh"], rng),
            "alpha": random_log_float(1e-5, 1e-1, rng),
            "learning_rate_init": random_log_float(1e-4, 1e-2, rng),
            "batch_size": int(random_choice([32, 64, 128], rng)),
            "early_stopping": bool(random_choice([True, False], rng)),
        })
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return params


def suggest_model_params(model_name, trial, center_params):
    params = {
        "sampling_strategy": trial.suggest_categorical(
            "sampling_strategy",
            categorical_window(center_params["sampling_strategy"], smote_strategies, radius=1),
        ),
        "k_neighbors": trial.suggest_int(
            "k_neighbors",
            *int_bounds_around(center_params["k_neighbors"], 1, 7, radius=2),
        ),
    }

    if model_name == "RandomForest":
        params.update({
            "n_estimators": trial.suggest_int(
                "n_estimators",
                *int_bounds_around(center_params["n_estimators"], 100, 1000, radius=200),
                step=100,
            ),
            "max_depth": trial.suggest_categorical(
                "max_depth",
                categorical_window(center_params["max_depth"], [3, 5, 8, 12, None], radius=1),
            ),
            "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf",
                *int_bounds_around(center_params["min_samples_leaf"], 1, 80, radius=15),
                log=True,
            ),
            "min_samples_split": trial.suggest_int(
                "min_samples_split",
                *int_bounds_around(center_params["min_samples_split"], 2, 150, radius=30),
                log=True,
            ),
            "max_features": trial.suggest_categorical(
                "max_features",
                categorical_window(center_params["max_features"], ["sqrt", "log2", 0.5, None], radius=1),
            ),
            "bootstrap": trial.suggest_categorical("bootstrap", [center_params["bootstrap"]]),
        })
    elif model_name == "XGBoost":
        params.update({
            "n_estimators": trial.suggest_int(
                "n_estimators",
                *int_bounds_around(center_params["n_estimators"], 50, 1000, radius=200),
                step=50,
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                *int_bounds_around(center_params["max_depth"], 2, 10, radius=2),
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                *log_bounds_around(center_params["learning_rate"], 0.005, 0.30, factor=3.0),
                log=True,
            ),
            "min_child_weight": trial.suggest_float(
                "min_child_weight",
                *log_bounds_around(center_params["min_child_weight"], 0.5, 30.0, factor=3.0),
                log=True,
            ),
            "subsample": trial.suggest_float(
                "subsample",
                *float_bounds_around(center_params["subsample"], 0.50, 1.00, radius=0.15),
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                *float_bounds_around(center_params["colsample_bytree"], 0.50, 1.00, radius=0.15),
            ),
            "gamma": trial.suggest_float(
                "gamma",
                *float_bounds_around(center_params["gamma"], 0.0, 8.0, radius=2.0),
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                *log_bounds_around(center_params["reg_alpha"], 1e-5, 20.0, factor=5.0),
                log=True,
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                *log_bounds_around(center_params["reg_lambda"], 1e-3, 100.0, factor=5.0),
                log=True,
            ),
        })
    elif model_name == "LightGBM":
        params.update({
            "n_estimators": trial.suggest_int(
                "n_estimators",
                *int_bounds_around(center_params["n_estimators"], 50, 1000, radius=200),
                step=50,
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                *log_bounds_around(center_params["learning_rate"], 0.005, 0.30, factor=3.0),
                log=True,
            ),
            "num_leaves": trial.suggest_int(
                "num_leaves",
                *int_bounds_around(center_params["num_leaves"], 4, 96, radius=20),
            ),
            "max_depth": trial.suggest_categorical(
                "max_depth",
                categorical_window(center_params["max_depth"], [2, 3, 4, 5, 7, -1], radius=1),
            ),
            "min_child_samples": trial.suggest_int(
                "min_child_samples",
                *int_bounds_around(center_params["min_child_samples"], 10, 200, radius=40),
            ),
            "subsample": trial.suggest_float(
                "subsample",
                *float_bounds_around(center_params["subsample"], 0.50, 1.00, radius=0.15),
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                *float_bounds_around(center_params["colsample_bytree"], 0.50, 1.00, radius=0.15),
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                *log_bounds_around(center_params["reg_alpha"], 1e-5, 20.0, factor=5.0),
                log=True,
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                *log_bounds_around(center_params["reg_lambda"], 1e-3, 100.0, factor=5.0),
                log=True,
            ),
        })
    elif model_name == "MLP":
        params.update({
            "hidden_layer_sizes": trial.suggest_categorical(
                "hidden_layer_sizes",
                categorical_window(center_params["hidden_layer_sizes"], list(mlp_architectures), radius=1),
            ),
            "activation": trial.suggest_categorical("activation", [center_params["activation"]]),
            "alpha": trial.suggest_float(
                "alpha",
                *log_bounds_around(center_params["alpha"], 1e-6, 1.0, factor=5.0),
                log=True,
            ),
            "learning_rate_init": trial.suggest_float(
                "learning_rate_init",
                *log_bounds_around(center_params["learning_rate_init"], 1e-5, 5e-2, factor=3.0),
                log=True,
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size",
                categorical_window(center_params["batch_size"], [32, 64, 128], radius=1),
            ),
            "early_stopping": trial.suggest_categorical("early_stopping", [center_params["early_stopping"]]),
        })
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return params


def build_estimator(model_name, params):
    if model_name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            min_samples_split=params["min_samples_split"],
            max_features=params["max_features"],
            bootstrap=params["bootstrap"],
            random_state=42,
            n_jobs=-1,
        )
    if model_name == "XGBoost":
        return XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            min_child_weight=params["min_child_weight"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            gamma=params["gamma"],
            reg_alpha=params["reg_alpha"],
            reg_lambda=params["reg_lambda"],
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
        )
    if model_name == "LightGBM":
        return LGBMClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            num_leaves=params["num_leaves"],
            max_depth=params["max_depth"],
            min_child_samples=params["min_child_samples"],
            subsample=params["subsample"],
            subsample_freq=1,
            colsample_bytree=params["colsample_bytree"],
            reg_alpha=params["reg_alpha"],
            reg_lambda=params["reg_lambda"],
            objective="multiclass",
            num_class=len(class_ids),
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
    if model_name == "MLP":
        return SklearnPipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(
                hidden_layer_sizes=mlp_architectures[params["hidden_layer_sizes"]],
                activation=params["activation"],
                alpha=params["alpha"],
                learning_rate_init=params["learning_rate_init"],
                batch_size=params["batch_size"],
                solver="adam",
                max_iter=500,
                early_stopping=params["early_stopping"],
                validation_fraction=0.15,
                n_iter_no_change=20,
                random_state=42,
            )),
        ])
    raise ValueError(f"Unknown model name: {model_name}")


def build_smote_classifier(model_name, params):
    return make_smote_classifier(
        estimator=build_estimator(model_name, params),
        sampling_strategy=params["sampling_strategy"],
        k_neighbors=params["k_neighbors"],
        random_state=42,
    )


def evaluate_params(model_name, params, include_test=False):
    model = build_smote_classifier(model_name, params)
    model.fit(X_train, y_train)
    train_scores = score_classifier(model, X_train, y_train)
    val_scores = score_classifier(model, X_val, y_val)
    row = {
        "model": model_name,
        "sampling_strategy": params["sampling_strategy"],
        "k_neighbors": params["k_neighbors"],
        "train_macro_recall": train_scores["macro_recall"],
        "validation_macro_recall": val_scores["macro_recall"],
        "train_macro_f1": train_scores["macro_f1"],
        "validation_macro_f1": val_scores["macro_f1"],
        "train_validation_macro_recall_gap": train_scores["macro_recall"] - val_scores["macro_recall"],
        "validation_accuracy": val_scores["accuracy"],
        "class_counts_before": model.class_counts_before_,
        "class_counts_after": model.class_counts_after_,
        "params": params,
    }
    if include_test:
        test_scores = score_classifier(model, X_test, y_test)
        row.update({
            "test_macro_recall": test_scores["macro_recall"],
            "test_macro_f1": test_scores["macro_f1"],
            "train_test_macro_recall_gap": train_scores["macro_recall"] - test_scores["macro_recall"],
            "test_accuracy": test_scores["accuracy"],
        })
    return row, model


def make_local_objective(model_name, center_params):
    def objective(trial):
        params = suggest_model_params(model_name, trial, center_params)
        try:
            row, _ = evaluate_params(model_name, params, include_test=False)
        except Exception as exc:
            raise optuna.TrialPruned(str(exc)) from exc
        return row["validation_macro_recall"]
    return objective


def rounded_display(frame):
    metric_columns = [
        column for column in frame.columns
        if column.endswith(("recall", "f1", "accuracy", "gap")) or column == "best_value"
    ]
    return frame.assign(**{column: frame[column].round(4) for column in metric_columns})


## 4. Run Random Search and Local Optuna Search

Increase `n_random_trials_per_model` and `n_optuna_trials_per_model` for a more serious run. The current values are intended for rough exploration, not final reporting. MLP trials may take longer than the tree-model trials, so reduce these values if you only need a quick smoke test.


In [ ]:
model_names = ["RandomForest", "XGBoost", "LightGBM", "MLP"]
n_random_trials_per_model = 30
n_optuna_trials_per_model = 50
rng = np.random.default_rng(42)

random_search_rows = []

for model_name in model_names:
    for trial_number in range(n_random_trials_per_model):
        params = sample_random_model_params(model_name, rng)
        try:
            row, _ = evaluate_params(model_name, params, include_test=False)
            row.update({
                "search_stage": "random",
                "trial_number": trial_number,
                "status": "ok",
            })
        except Exception as exc:
            row = {
                "model": model_name,
                "search_stage": "random",
                "trial_number": trial_number,
                "status": f"failed: {type(exc).__name__}: {exc}",
                "params": params,
            }
        random_search_rows.append(row)

random_results_df = pd.DataFrame(random_search_rows)
ok_random_results = random_results_df.query("status == 'ok'").copy()

if ok_random_results.empty:
    raise RuntimeError("All random-search trials failed. Check the parameter search spaces.")

best_random_indices = ok_random_results.groupby("model")["validation_macro_recall"].idxmax()
best_random_df = (
    ok_random_results.loc[best_random_indices]
    .sort_values("validation_macro_recall", ascending=False)
    .reset_index(drop=True)
)
best_random_params = dict(zip(best_random_df["model"], best_random_df["params"]))

display(
    rounded_display(
        ok_random_results
        .sort_values("validation_macro_recall", ascending=False)
        .head(20)
    )
)
display(rounded_display(best_random_df))

studies = {}
best_models = {}
optuna_rows = []

for model_name in model_names:
    center_params = best_random_params[model_name]
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name=f"{model_name}_whc_local_optuna_search",
    )
    study.optimize(
        make_local_objective(model_name, center_params),
        n_trials=n_optuna_trials_per_model,
        show_progress_bar=True,
    )
    studies[model_name] = study

    row, best_model = evaluate_params(model_name, study.best_params, include_test=True)
    row.update({
        "search_stage": "local_optuna",
        "best_value": study.best_value,
        "best_random_params": center_params,
        "best_params": study.best_params,
    })
    optuna_rows.append(row)
    best_models[model_name] = best_model

results_df = pd.DataFrame(optuna_rows).sort_values("validation_macro_recall", ascending=False)
display(rounded_display(results_df))


/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml

,model,sampling_strategy,k_neighbors,train_macro_recall,validation_macro_recall,train_macro_f1,validation_macro_f1,train_validation_macro_recall_gap,validation_accuracy,class_counts_before,class_counts_after,params,search_stage,trial_number,status
14,RandomForest,auto,2,0.6960,0.3079,0.3709,0.2773,0.3881,0.4479,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 2...",random,14,ok
32,XGBoost,auto,3,0.9202,0.3012,0.7208,0.2999,0.6190,0.5370,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 3...",random,2,ok
5,RandomForest,strong,2,0.7668,0.2997,0.5064,0.2705,0.4671,0.4726,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 3421, 2: 3421, 3: 3421}","{'sampling_strategy': 'strong', 'k_neighbors':...",random,5,ok
18,RandomForest,auto,2,0.8977,0.2980,0.7477,0.2981,0.5997,0.5425,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 2...",random,18,ok
42,XGBoost,auto,1,0.6011,0.2976,0.5868,0.2946,0.3035,0.5425,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 1...",random,12,ok
55,XGBoost,auto,2,0.7923,0.2960,0.5549,0.2911,0.4963,0.5274,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 2...",random,25,ok
78,LightGBM,moderate,5,0.9534,0.2956,0.9607,0.2928,0.6578,0.5397,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 3234, 2: 2281, 3: 2281}","{'sampling_strategy': 'moderate', 'k_neighbors...",random,18,ok
65,LightGBM,auto,5,0.9658,0.2941,0.9618,0.2962,0.6716,0.5233,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 5...",random,5,ok
36,XGBoost,auto,1,0.4796,0.2940,0.3063,0.2591,0.1855,0.4082,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 1...",random,6,ok
48,XGBoost,strong,2,0.6749,0.2919,0.4324,0.2783,0.3830,0.5082,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 3421, 2: 3421, 3: 3421}","{'sampling_strategy': 'strong', 'k_neighbors':...",random,18,ok


,model,sampling_strategy,k_neighbors,train_macro_recall,validation_macro_recall,train_macro_f1,validation_macro_f1,train_validation_macro_recall_gap,validation_accuracy,class_counts_before,class_counts_after,params,search_stage,trial_number,status
0,RandomForest,auto,2,0.6960,0.3079,0.3709,0.2773,0.3881,0.4479,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 2...",random,14,ok
1,XGBoost,auto,3,0.9202,0.3012,0.7208,0.2999,0.6190,0.5370,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 3...",random,2,ok
2,LightGBM,moderate,5,0.9534,0.2956,0.9607,0.2928,0.6578,0.5397,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 3234, 2: 2281, 3: 2281}","{'sampling_strategy': 'moderate', 'k_neighbors...",random,18,ok
3,MLP,auto,7,0.7978,0.2848,0.6203,0.2832,0.5130,0.5055,"{0: 4561, 1: 3234, 2: 203, 3: 8}","{0: 4561, 1: 4561, 2: 4561, 3: 4561}","{'sampling_strategy': 'auto', 'k_neighbors': 7...",random,10,ok


  0%|          | 0/50 [00:00<?, ?it/s]/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ratnadeeppatra/adv_ml/36120-26SP-AT2-26294153-experiments/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:205: RuntimeWarning: overflow encountered in matmul
  re

## 5. Inspect Best Local Optuna Model


In [ ]:
best_row = results_df.iloc[0]
best_model_name = best_row["model"]
best_model = best_models[best_model_name]

print("Best validation model:", best_model_name)
display(best_row.to_frame("value"))

for split_name, X, y in [
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test),
]:
    predictions = best_model.predict(X)
    print(f"\n{split_name} classification report")
    print(classification_report(
        y,
        predictions,
        labels=class_ids,
        target_names=class_names,
        zero_division=0,
    ))


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, split_name, X, y in [
    (axes[0], "Validation", X_val, y_val),
    (axes[1], "Test", X_test, y_test),
]:
    predictions = best_model.predict(X)
    matrix = confusion_matrix(y, predictions, labels=class_ids)
    sns.heatmap(matrix, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(title=f"{split_name} Confusion Matrix", xlabel="Predicted", ylabel="Actual")

figure.tight_layout()


## 6. Overfitting Check

Use the train-validation and train-test gaps to judge whether the random-search plus local-Optuna process found a genuinely useful model or only a high-capacity classifier that memorised the training period. A large positive gap means the model should be regularised further or rejected as a formal experiment candidate.


In [ ]:
overfit_diagnostic = results_df.copy()
overfit_diagnostic["overfitting_flag"] = overfit_diagnostic["train_validation_macro_recall_gap"].gt(0.30)
overfit_columns = [
    "model",
    "sampling_strategy",
    "k_neighbors",
    "train_macro_recall",
    "validation_macro_recall",
    "test_macro_recall",
    "train_validation_macro_recall_gap",
    "train_test_macro_recall_gap",
    "overfitting_flag",
]
display(
    overfit_diagnostic[overfit_columns]
    .sort_values("train_validation_macro_recall_gap", ascending=False)
    .round(4)
)


## 7. Rough Conclusion

Compare the local-Optuna-selected models against the logistic-regression experiment using validation macro recall first, then test macro recall and macro F1. If local Optuna improves validation performance but creates a large train-validation gap, treat that result as overfitting rather than a reliable improvement.
